# Track 1: Quantum Simulation of Photo-Induced Charge Dynamics ( From Ethylene to Photosynthesis )

## Task 1: Constructing the Ethylene Hamiltonian and Mapping to Qubits

### Ethylene Geometry at Equilibrium

```

 H       H 
  \     /  
   C = C   
  /     \  
 H       H 

```
C=C Bond Length $\approx 1.34Å$ 

C-H Bond Length $\approx 1.087Å$

H-C-H Bond Angle $\approx 117.3^{\circ}$

Carbons are relatively positioned at $ x=0, y=0 $ and aligned along z-axis where the distance of each from the origin is equal in value.

Carbon $ |d_z| = \frac{1.34}{2}Å = 0.67Å $

At $\pm0.67$ on the z-axis where the carbon atoms are located, bonds with the hydrogen atoms form. These bonds are tilted by an angle. We calculate the displacements on $y, z$ axes.

Hydrogen $|d_y| = 1.087\sin(\frac{117.3}{2}) = 0.928Å$

Hydrogen $|d_z| = 1.087\cos(\frac{117.3}{2}) + $ Carbon $|d_z| = 0.5655Å + 0.67Å = 1.2355Å$

Meanwhile there is no displacement on the x axis because it is planar.

Now we can finally construct the PySCF driver for Ethylene at equilibrium as follows:

In [33]:
from qiskit_nature.second_q.drivers import PySCFDriver

eth_mol = PySCFDriver(
    atom="C 0 0 0.67; H 0 0.928 1.2355; H 0 -0.928 1.2355; C 0 0 -0.67; H 0 0.928 -1.2355; H 0 -0.928 -1.2355",
    basis='sto3g'
)

es_problem = eth_mol.run()

### Ethylene Geometry at a Twisted $90^{\circ}$ Angle ($CH_2$ Groups Perpendicular to Eachother)

```
  H H     
  |  \    
  C - C   
  |    \  
  H     H 
```

Twisting one of the $CH_2$ groups $90^\circ$ isn't merely a change in the axes,  such alteration eliminates the spatial overlap which breaks the  $\pi$ bond into a relatively longer $\sigma$ bond. It also causes change in the C-H bond length and the bond angle because of the general change in the electronic structure.

The approximated new lengths and angle are as follows:

C-C Bond Length $\approx 1.46Å$ 

C-H Bond Length $\approx 1.08Å$

H-C-H Bond Angle $\approx 121^{\circ}$

Carbon $|d_z| = 1.46/2 = 0.73Å$

But now after the twist, hydrogens are on the x-axis instead of y where the new displacement on the x-axis is equal to the one on the y-axis at equilibrium, while remaining on the z-axis as well

Hydrogen $|d_x| = 0.928$

Hydrogen $|d_z| = 1.08\cos(\frac{121}{2}) +$ Carbon $ |d_z| = 0.53Å + 0.73Å = 1.26Å$



In [34]:
eth_mol_twisted = PySCFDriver(
    atom="C 0 0 0.73; H 0.928 0 1.26; H -0.928 0 1.26; C 0 0 -0.73; H 0.928 0 -1.26; H -0.928 0 -1.26",
    basis='sto3g'
)

es_problem_twisted = eth_mol_twisted.run()

### Reducing to Active Space

2 electrons, 2 orbitals expandable to 2 electrons, 4 orbitals

In [35]:
n_e = 2
n_orb = 2 # change to 4 for expansion

from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

transformer = ActiveSpaceTransformer(num_electrons=n_e,num_spatial_orbitals=n_orb)

reduced_problem = transformer.transform(es_problem)
reduced_problem_twisted = transformer.transform(es_problem_twisted)

### Mapping to Qubits and Getting Hamiltonian for Ethylene in Equilibrium and Twisted Geometry

In [36]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()
h = mapper.map(reduced_problem.second_q_ops()[0]) # index of the main hamiltonian in returned tuple
h_twisted = mapper.map(reduced_problem_twisted.second_q_ops()[0])

In [37]:
print(h)
print(h_twisted)

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.67810865+0.j,  0.07683157+0.j, -0.07802605+0.j,  0.08418098+0.j,
  0.07683157+0.j,  0.12685108+0.j, -0.07802605+0.j,  0.12720766+0.j,
  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,
  0.12720766+0.j,  0.13086927+0.j,  0.08418098+0.j])
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.65809122+0.j,  0.06206056+0.j, -0.0626729 +0.j,  0.07940855+0.j,
  0.06206056+0.j,  0.12362153+0.j, -0.0626729 +0.j,  0.12451616+0.j,
  0.04510761+0.j,  0.04510761+0.j,  0.04510761+0.j,  0.04510761+0.j,
  0.12451616+0.j,  0.12784495+0.j,  0.07940855+0.j])


**Now that we constructed the ethylene Hamiltonian in both geometries, for the next tasks we will only focus on ethylene at equilibrium.**

## Task 2: Ground State via VQE

### Bulding UCCSD Ansatz

In order to build a UCCSD ansatz we must start from a Hartree-Fock initial state

In [38]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

num_spatial_orbitals = reduced_problem.num_spatial_orbitals
num_particles = reduced_problem.num_particles

initial_state = HartreeFock(num_spatial_orbitals, num_particles, mapper)

uccsd_ansatz = UCCSD(num_spatial_orbitals, num_particles, mapper, initial_state = initial_state)

### Running VQE for UCCSD Ansatz

In [39]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector

estimator = StatevectorEstimator()

uccsd_vqe = VQE(estimator=estimator, ansatz=uccsd_ansatz, optimizer=COBYLA())

uccsd_result = uccsd_vqe.compute_minimum_eigenvalue(operator=h)

optimal_uccsd = uccsd_ansatz.assign_parameters(uccsd_result.optimal_parameters)

uccsd_gse = uccsd_result.eigenvalue.real # UCCSD ground state energy

uccsd_gsv = Statevector(optimal_uccsd) # UCCSD ground state vector

/home/rabee/.qiskit/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/home/rabee/.qiskit/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


### Building Hardware Efficient Ansatz

In [40]:
from qiskit.circuit.library import efficient_su2

num_qubits = h.num_qubits # = 2 * num_spatial_orbitals

hea = efficient_su2(num_qubits)

### Running VQE for HEA

In [41]:
hea_vqe = VQE(estimator=estimator, ansatz=hea, optimizer=COBYLA())

hea_result = hea_vqe.compute_minimum_eigenvalue(operator=h)

optimal_hea = hea.assign_parameters(hea_result.optimal_parameters)

hea_gse = hea_result.eigenvalue.real # HEA ground state energy

hea_gsv = Statevector(optimal_hea) # HEA ground state vector

## Task 3: Excited states via QSE

In [42]:
hamiltonian = h
import numpy as np
from scipy.linalg import eigh
from qiskit_nature.second_q.operators import FermionicOp

# Build spin-adapted singlet excitation pool
n_spin = 4

# Identity operator (ground state reference)
op_identity = FermionicOp({"": 1.0}, num_spin_orbitals=n_spin)

# Singlet Single Excitation Generator: 1/sqrt(2) * (E_alpha + E_beta)
# E_alpha = a_2^\dagger a_0 - a_0^\dagger a_2
# E_beta  = a_3^\dagger a_1 - a_1^\dagger a_3
op_singlet_single = FermionicOp(
    {
        "+_2 -_0": 1.0 / np.sqrt(2),
        "+_0 -_2": -1.0 / np.sqrt(2),
        "+_3 -_1": 1.0 / np.sqrt(2),
        "+_1 -_3": -1.0 / np.sqrt(2),
    },
    num_spin_orbitals=n_spin,
)

# Singlet Double Excitation Generator
op_singlet_double = FermionicOp(
    {"+_2 +_3 -_1 -_0": 1.0, "+_0 +_1 -_3 -_2": -1.0}, num_spin_orbitals=n_spin
)

# Pool containing spin-adapted singlet operators
fermi_pool = [op_identity, op_singlet_single, op_singlet_double]

# Map to Pauli operators
mapper = JordanWignerMapper()
pauli_pool = [mapper.map(op) for op in fermi_pool]

print(f"Number of operators: {len(pauli_pool)}")
for i, op in enumerate(pauli_pool):
    print(f"O_{i}:\n{op}\n")

Number of operators: 3
O_0:
SparsePauliOp(['IIII'],
              coeffs=[1.+0.j])

O_1:
SparsePauliOp(['IXZY', 'IYZX', 'XZYI', 'YZXI'],
              coeffs=[0.+0.35355339j, 0.-0.35355339j, 0.+0.35355339j, 0.-0.35355339j])

O_2:
SparsePauliOp(['XYXX', 'XYYY', 'YYXY', 'YYYX', 'XXXY', 'XXYX', 'YXXX', 'YXYY'],
              coeffs=[0.-0.125j, 0.+0.125j, 0.-0.125j, 0.-0.125j, 0.+0.125j, 0.+0.125j,
 0.-0.125j, 0.+0.125j])



In [43]:
# Build H and S matrices
n_ops = len(pauli_pool)
H_mat = np.zeros((n_ops, n_ops), dtype=complex)
S_mat = np.zeros((n_ops, n_ops), dtype=complex)

for i in range(n_ops):
    Oi = pauli_pool[i]
    Oi_dag = Oi.conjugate().transpose()
    for j in range(n_ops):
        Oj = pauli_pool[j]

        # Overlap matrix: <Ψ| Oi† Oj |Ψ>
        overlap_op = Oi_dag @ Oj
        S_val = uccsd_gsv.expectation_value(overlap_op)
        S_mat[i, j] = S_val

        # Hamiltonian matrix: <Ψ| Oi† H Oj |Ψ>
        H_Oj = hamiltonian @ Oj
        full_op = Oi_dag @ H_Oj
        H_val = uccsd_gsv.expectation_value(full_op)
        H_mat[i, j] = H_val

# Take real parts (should be Hermitian)
S_real = np.real(S_mat)
H_real = np.real(H_mat)

print("S matrix (should have 1 on diagonal for identity):")
print(S_real)
print("\nH matrix:")
print(H_real)

S matrix (should have 1 on diagonal for identity):
[[1.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 1.89607452e-09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00]]

H matrix:
[[-1.19748783e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -1.93756510e-09  0.00000000e+00]
 [-1.73472348e-18  0.00000000e+00  0.00000000e+00]]


In [44]:
# Verification
print(f"S is Hermitian: {np.allclose(S_real, S_real.T)}")
S_eigvals = np.linalg.eigvalsh(S_real)
print(f"Smallest eigenvalue of S: {S_eigvals.min():.6f} (should be > 0)")

# Check that identity operator gives correct energy
# The first row/column should correspond to identity
# The (0,0) element of H should equal VQE energy
print(f"\nH[0,0] (from identity-identity): {H_real[0,0]:.8f} Ha")
print(f"VQE energy: {uccsd_gse:.8f} Ha")
print(f"Difference: {abs(H_real[0,0] - uccsd_gse):.2e} Ha")

S is Hermitian: True
Smallest eigenvalue of S: 0.000000 (should be > 0)

H[0,0] (from identity-identity): -1.19748783 Ha
VQE energy: -1.19748783 Ha
Difference: 2.22e-16 Ha


In [45]:
# --- Solve H v = E S v ---
try:
    eigvals, eigvecs = eigh(H_real, S_real)
except np.linalg.LinAlgError as e:
    print("S is singular. Adding small regularization...")
    S_reg = S_real + 1e-10 * np.eye(len(S_real))
    eigvals, eigvecs = eigh(H_real, S_reg)

print("\nEigenvalues (Hartree):")
for i, val in enumerate(eigvals):
    print(f"  E_{i} = {val:.8f} Ha")

S is singular. Adding small regularization...

Eigenvalues (Hartree):
  E_0 = -1.19748783 Ha
  E_1 = -0.97068776 Ha
  E_2 = 0.00000000 Ha


In [46]:
# Extract excitation energy
E0_qse = eigvals[0]
E1_qse = eigvals[1] if len(eigvals) > 1 else None

print(f"QSE ground energy: {E0_qse:.8f} Ha")
print(f"VQE ground energy: {uccsd_gse:.8f} Ha")
print(f"Difference: {abs(E0_qse - uccsd_gse):.2e} Ha")

if E1_qse is not None:
    delta_E_ha = E1_qse - E0_qse
    delta_E_ev = delta_E_ha * 27.2114
    print(f"\nFirst excited energy: {E1_qse:.8f} Ha")
    print(f"Vertical excitation energy: {delta_E_ev:.4f} eV")
    print(f"Literature value: ~7.6 eV")
    print(f"Deviation: {abs(delta_E_ev - 7.6):.4f} eV")
else:
    print("Only one eigenvalue found – need more operators.")

QSE ground energy: -1.19748783 Ha
VQE ground energy: -1.19748783 Ha
Difference: 1.20e-10 Ha

First excited energy: -0.97068776 Ha
Vertical excitation energy: 6.1715 eV
Literature value: ~7.6 eV
Deviation: 1.4285 eV
